In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/data_cleaned.csv")
df.head()

,name,price,year,status,origin,mileage,fuel_type,body_type,brand,age
0,Mercedes-Benz GLC 200 4Matic 2021,1389000000,2021,Xe cũ,Trong nước,55555.0,xăng,Other,Mercedes,4
1,Kia Sonet 1.5 Premium 2024,585000000,2024,Xe cũ,Trong nước,1000.0,xăng,Crossover,Kia,1
2,Ford Everest Titanium 2.0L AT 4WD 2019,810000000,2019,Xe cũ,Nhập khẩu,4.0,xăng,SUV,Ford,6
3,Mercedes-Benz GLC 300 4Matic 2016,885000000,2016,Xe cũ,Nhập khẩu,90000.0,xăng,SUV,Mercedes,9
4,Mazda 6 2.0 AT 2017,440000000,2017,Xe cũ,Trong nước,57225.0,xăng,Sedan,Mazda,8


In [4]:
df = df.drop(["name","year"],axis=1)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1342 entries, 0 to 1341
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Body_Type     1342 non-null   object
 1   Origin        1342 non-null   object
 2   Province      1342 non-null   object
 3   District      1342 non-null   object
 4   Transmission  1342 non-null   object
 5   Fuel_Type     1342 non-null   object
 6   Brand         1342 non-null   object
 7   price_num     1342 non-null   int64 
 8   age           1342 non-null   int64 
 9   mileage_num   1342 non-null   int64 
dtypes: int64(3), object(7)
memory usage: 105.0+ KB


In [5]:
X = df.drop(["price"],axis=1)

y = df["price"]

In [6]:
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')


categorical_cols = [col for col in X.columns if X[col].dtype == 'object']
numerical_cols = [col for col in X.columns if col not in categorical_cols and X[col].dtype in ['float64', 'int64']]


ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat = ohe.fit_transform(X[categorical_cols])
cat_feature_names = ohe.get_feature_names_out(categorical_cols)
X_cat_df = pd.DataFrame(X_cat, columns=cat_feature_names, index=X.index)


scaler = MinMaxScaler()
X_num = scaler.fit_transform(X[numerical_cols])
X_num_df = pd.DataFrame(X_num, columns=numerical_cols, index=X.index)


X_encoded = pd.concat([X_num_df, X_cat_df], axis=1)
X_encoded.shape

(3840, 72)

In [7]:
from sklearn.model_selection import train_test_split

np.random.seed(42)

X_train, X_test, y_train, y_test =  train_test_split(X_encoded,
                                                     y,
                                                     test_size=0.2)

In [25]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import cross_val_score


models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "Random Forest": RandomForestRegressor()
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results[name] = {"RMSE": rmse, "R2": r2}


for name, res in results.items():
    print(f"{name}: R2 Score = {res['R2']:.4f}, RMSE = {res['RMSE']:.2f}")

Linear Regression: R2 Score = 0.7037, RMSE = 271195339.63
Ridge Regression: R2 Score = 0.7020, RMSE = 271953463.66
Lasso Regression: R2 Score = 0.7037, RMSE = 271195347.77
Random Forest: R2 Score = 0.8215, RMSE = 210510191.49


In [26]:
LR = LinearRegression()
Rid = Ridge()
Las = Lasso()
Rfr = RandomForestRegressor()

In [27]:
LR.fit(X_train,y_train)

cvs = cross_val_score(LR, X_encoded, y, cv=5)
print(cvs)
print(cvs.mean())

[0.65549607 0.67906559 0.63279839 0.61938799 0.62630777]
0.6426111608089135


In [28]:
Rid.fit(X_train,y_train)

cvs = cross_val_score(Rid, X_encoded, y, cv=5)
print(cvs)
print(cvs.mean())

[0.65559091 0.68364698 0.66980285 0.63137409 0.62566079]
0.65321512312898


In [29]:
Las.fit(X_train,y_train)

cvs = cross_val_score(Las, X_encoded, y, cv=5)

print(cvs)
print(cvs.mean())

[0.65613361 0.68056847 0.63360832 0.61605914 0.62682689]
0.6426392836767367


In [30]:
Rfr.fit(X_train,y_train)

cvs = cross_val_score(Rfr, X_encoded, y, cv=5)

print(cvs)
print(cvs.mean())

[0.79132408 0.76773361 0.77020687 0.72567112 0.78494576]
0.7679762876803028


In [31]:
from sklearn.model_selection import GridSearchCV

params_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_rf = GridSearchCV(Rfr, params_rf, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_rf.fit(X_train, y_train)


GridSearchCV(cv=3, estimator=RandomForestRegressor(), n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20, 30],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='neg_mean_squared_error')

In [32]:
grid_rf.best_params_

{'max_depth': 30,
 'min_samples_leaf': 1,
 'min_samples_split': 10,
 'n_estimators': 200}

In [33]:
rfr_fix = RandomForestRegressor(
    max_depth = 20,
    min_samples_leaf = 1,
    min_samples_split = 5,
    n_estimators = 200
)

rfr_fix.fit(X_train,y_train)

rfr_fix.score(X_test,y_test)

0.8252830646511125

In [34]:
cvs = cross_val_score(rfr_fix, X_encoded, y, cv=5)

print(cvs)
print(cvs.mean())

[0.78361874 0.77166842 0.7805753  0.73444637 0.78487694]
0.7710371527448918


In [35]:
from sklearn.ensemble import RandomForestRegressor

params_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_rf = GridSearchCV(Rfr, params_rf, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
grid_rf.fit(X_train, y_train)


GridSearchCV(cv=3, estimator=RandomForestRegressor(), n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20, 30],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='neg_mean_squared_error')

In [36]:
best_params = grid_rf.best_params_
print("Best Parameters:", best_params)

Best Parameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 300}


In [37]:
rfr_fix = RandomForestRegressor(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    min_samples_leaf=best_params['min_samples_leaf'],
    random_state=42
)

In [38]:
rfr_fix.fit(X_train, y_train)

RandomForestRegressor(max_depth=20, min_samples_split=5, n_estimators=300,
                      random_state=42)

In [39]:
train_score = rfr_fix.score(X_train, y_train)
test_score = rfr_fix.score(X_test, y_test)

print(f"Train R²: {train_score:.4f}")
print(f"Test  R²: {test_score:.4f}")

Train R²: 0.9181
Test  R²: 0.8244


In [40]:
cvs = cross_val_score(rfr_fix, X_encoded, y, cv=5)

print(cvs)
print(cvs.mean())

[0.78790784 0.77506156 0.7790241  0.73410229 0.78265045]
0.7717492478025139


In [41]:
from joblib import dump

dump(rfr_fix, "../model/random_forest_model_1.joblib")
dump(ohe, "../model/onehot_encoder.pkl")
dump(scaler,"../model/scaler.pkl")


['../model/scaler.pkl']